Célula 1 — Instalação (mantém essa, é única)

In [1]:
# ── Instalação ────────────────────────────────────────────
!apt-get install -y tesseract-ocr tesseract-ocr-por -q
!pip install pytesseract pdf2image Pillow spacy nltk scikit-learn gradio -q

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
!python -m spacy download pt_core_news_sm -q

print("✅ Tudo instalado!")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-por
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 856 kB of archives.
After this operation, 1,998 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-por all 1:4.00~git30-7274cfa-1.1 [856 kB]
Fetched 856 kB in 1s (923 kB/s)
Selecting previously unselected package tesseract-ocr-por.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-por_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-por (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-por (1:4.00~git30-7274cfa-1.1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 35.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the pac

Célula 2 — Criar pastas (mantém essa)

In [2]:
# ── Estrutura de pastas ───────────────────────────────────
import os

for p in ["cv-analyzer/backend", "cv-analyzer/assets",
          "cv-analyzer/notebooks", "cv-analyzer/utils",
          "cv-analyzer/data/raw", "cv-analyzer/data/processed"]:
    os.makedirs(p, exist_ok=True)
    print(f"✅ {p}")

✅ cv-analyzer/backend
✅ cv-analyzer/assets
✅ cv-analyzer/notebooks
✅ cv-analyzer/utils
✅ cv-analyzer/data/raw
✅ cv-analyzer/data/processed


Célula 3 — Escrever ocr_module.py

In [3]:
%%writefile cv-analyzer/backend/ocr_module.py
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import re

def extrair_texto(caminho_arquivo: str) -> str:
    ext = caminho_arquivo.lower().split('.')[-1]
    print(f"📄 Processando: {caminho_arquivo}")

    if ext == 'pdf':
        paginas = convert_from_path(caminho_arquivo, dpi=300)
        return "\n".join(
            pytesseract.image_to_string(p, lang='por+eng', config='--psm 6')
            for p in paginas
        ).strip()

    elif ext in ['jpg', 'jpeg', 'png']:
        img = Image.open(caminho_arquivo).convert('L')
        return pytesseract.image_to_string(img, lang='por+eng', config='--psm 6').strip()

    return "❌ Formato não suportado. Use PDF, JPG ou PNG."

Writing cv-analyzer/backend/ocr_module.py


Célula 4 — Escrever nlp_module.py

In [4]:
%%writefile cv-analyzer/backend/nlp_module.py
import spacy, re
from nltk.corpus import stopwords

nlp = spacy.load("pt_core_news_sm")

HABILIDADES_TECNICAS = {
    "python","java","javascript","typescript","c++","c#","go","rust",
    "kotlin","swift","php","ruby","scala","django","flask","fastapi",
    "react","vue","angular","node","spring","laravel","rails",
    "postgresql","mysql","mongodb","redis","sqlite","oracle",
    "docker","kubernetes","aws","azure","gcp","git","github",
    "jenkins","terraform","ansible","pandas","numpy","scikit-learn",
    "tensorflow","pytorch","spark","hadoop","power bi","tableau",
    "sql","nosql","rest","api","microservices","agile","scrum",
}

def limpar_texto(texto):
    texto = re.sub(r'\s+', ' ', texto)
    texto = re.sub(r'[^\w\s@.\-,/()áéíóúâêîôûãõàèìòùç]', '', texto, flags=re.IGNORECASE)
    return texto.strip()

def extrair_email(texto):
    r = re.findall(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', texto)
    return r[0] if r else "Não encontrado"

def extrair_telefone(texto):
    r = re.findall(r'(\(?\d{2}\)?\s?\d{4,5}[-\s]?\d{4})', texto)
    return r[0] if r else "Não encontrado"

def extrair_habilidades(texto):
    tl = texto.lower()
    return sorted([h for h in HABILIDADES_TECNICAS
                   if re.search(r'\b' + re.escape(h) + r'\b', tl)])

def extrair_entidades(texto):
    doc = nlp(texto[:5000])
    res = {"pessoas": [], "organizacoes": [], "datas": [], "locais": []}
    mapa = {"PER": "pessoas", "ORG": "organizacoes", "DATE": "datas",
            "LOC": "locais", "GPE": "locais"}
    for ent in doc.ents:
        if ent.label_ in mapa:
            res[mapa[ent.label_]].append(ent.text)
    for k in res:
        res[k] = list(set(res[k]))
    return res

def analisar_curriculo(texto):
    tl = limpar_texto(texto)
    return {
        "texto_limpo": tl,
        "email": extrair_email(tl),
        "telefone": extrair_telefone(tl),
        "habilidades": extrair_habilidades(tl),
        "entidades": extrair_entidades(tl),
    }

Writing cv-analyzer/backend/nlp_module.py


Célula 5 — Escrever ranker.py

In [5]:
%%writefile cv-analyzer/backend/ranker.py
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
import re

try:
    STOPWORDS_PT = set(stopwords.words('portuguese'))
except:
    STOPWORDS_PT = set()

def preprocessar(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', ' ', texto)
    return ' '.join([p for p in texto.split()
                     if p not in STOPWORDS_PT and len(p) > 2])

def calcular_score(texto_curriculo, texto_vaga, habilidades):
    c, v = preprocessar(texto_curriculo), preprocessar(texto_vaga)
    try:
        mat = TfidfVectorizer().fit_transform([c, v])
        sim_tfidf = float(cosine_similarity(mat[0:1], mat[1:2])[0][0])
    except:
        sim_tfidf = 0.0

    vl = texto_vaga.lower()
    em_comum = [h for h in habilidades if h in vl]
    extras   = [h for h in habilidades if h not in vl]
    pct_hab  = len(em_comum) / max(len(habilidades), 1)
    score    = round(min((sim_tfidf * 0.6 + pct_hab * 0.4) * 100, 100), 1)

    if score >= 75:   cls, rec = "🟢 Alta compatibilidade",       "Candidato fortemente recomendado!"
    elif score >= 50: cls, rec = "🟡 Média compatibilidade",       "Candidato com potencial."
    elif score >= 25: cls, rec = "🟠 Baixa compatibilidade",       "Candidato com lacunas relevantes."
    else:             cls, rec = "🔴 Muito baixa compatibilidade", "Perfil distante da vaga."

    return {
        "score": score, "classificacao": cls, "recomendacao": rec,
        "sim_textual": round(sim_tfidf * 100, 1),
        "match_hab": round(pct_hab * 100, 1),
        "em_comum": em_comum, "extras": extras,
    }

Writing cv-analyzer/backend/ranker.py


Célula 6 — Escrever app.py

In [6]:
%%writefile cv-analyzer/backend/app.py
import gradio as gr
from ocr_module import extrair_texto
from nlp_module import analisar_curriculo
from ranker import calcular_score

def processar(arquivo, descricao_vaga):
    if arquivo is None:            return "❌ Envie um currículo.", "", "", ""
    if not descricao_vaga.strip(): return "❌ Descreva a vaga.",    "", "", ""
    try:
        texto_bruto = extrair_texto(arquivo.name)
        if not texto_bruto:        return "❌ Não consegui extrair texto.", "", "", ""

        analise = analisar_curriculo(texto_bruto)
        r = calcular_score(analise["texto_limpo"], descricao_vaga, analise["habilidades"])

        principal = (
            f"## {r['classificacao']}\n"
            f"**Score:** {r['score']} / 100 — {r['recomendacao']}\n\n---\n"
            f"📊 Similaridade textual: **{r['sim_textual']}%** · "
            f"🛠️ Match de habilidades: **{r['match_hab']}%**"
        )
        habilidades = (
            f"**✅ Em comum com a vaga:** {', '.join(r['em_comum']) or 'Nenhuma'}\n\n"
            f"**➕ Extras do candidato:** {', '.join(r['extras']) or 'Nenhuma'}"
        )
        ent  = analise["entidades"]
        info = (
            f"**📧 E-mail:** {analise['email']}\n\n"
            f"**📞 Telefone:** {analise['telefone']}\n\n"
            f"**🏢 Organizações:** {', '.join(ent['organizacoes'][:5]) or 'Não identificadas'}\n\n"
            f"**📅 Datas:** {', '.join(ent['datas'][:5]) or 'Não identificadas'}"
        )
        ocr_preview = texto_bruto[:1500] + ("..." if len(texto_bruto) > 1500 else "")
        return principal, habilidades, info, ocr_preview

    except Exception as e:
        return f"❌ Erro: {e}", "", "", ""


with gr.Blocks(title="Analisador de Currículos", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🤖 Analisador de Currículos com OCR + IA")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📤 Entrada")
            arquivo_in = gr.File(label="Currículo (PDF, JPG ou PNG)",
                                 file_types=[".pdf", ".jpg", ".jpeg", ".png"])
            vaga_in    = gr.Textbox(label="Descrição da vaga",
                                    placeholder="Cole aqui os requisitos da vaga...",
                                    lines=8)
            btn        = gr.Button("🔍 Analisar", variant="primary", size="lg")
        with gr.Column():
            gr.Markdown("### 📊 Resultado")
            out_score = gr.Markdown()
            out_hab   = gr.Markdown()
            out_info  = gr.Markdown()
    with gr.Accordion("📄 Texto extraído pelo OCR", open=False):
        out_ocr = gr.Textbox(lines=10)
    btn.click(fn=processar,
              inputs=[arquivo_in, vaga_in],
              outputs=[out_score, out_hab, out_info, out_ocr])

if __name__ == "__main__":
    app.launch(share=True)

Writing cv-analyzer/backend/app.py


Célula 7 — Escrever requirements.txt e criar .gitkeep

In [7]:
%%writefile cv-analyzer/requirements.txt
pytesseract==0.3.10
pdf2image==1.16.3
Pillow==10.0.0
spacy==3.7.2
nltk==3.8.1
scikit-learn==1.3.0
gradio==4.7.1
numpy==1.24.0

Writing cv-analyzer/requirements.txt


In [8]:
# Cria marcadores para pastas vazias não serem ignoradas pelo Git
for caminho in ["cv-analyzer/data/raw/.gitkeep",
                "cv-analyzer/data/processed/.gitkeep",
                "cv-analyzer/assets/.gitkeep"]:
    open(caminho, 'w').close()
    print(f"✅ {caminho}")

✅ cv-analyzer/data/raw/.gitkeep
✅ cv-analyzer/data/processed/.gitkeep
✅ cv-analyzer/assets/.gitkeep


Célula 8 — Verificar estrutura

In [9]:
import os
for raiz, pastas, arquivos in os.walk("cv-analyzer"):
    nivel = raiz.replace("cv-analyzer", "").count(os.sep)
    print("  " * nivel + os.path.basename(raiz) + "/")
    for arquivo in arquivos:
        print("  " * (nivel + 1) + arquivo)

cv-analyzer/
  requirements.txt
  notebooks/
  assets/
    .gitkeep
  backend/
    nlp_module.py
    ranker.py
    app.py
    ocr_module.py
  data/
    processed/
      .gitkeep
    raw/
      .gitkeep
  utils/


Célula 9 — Lançar o Gradio e gerar o link público

In [12]:
# ── RODA A INTERFACE DIRETO NO COLAB ─────────────────────
import sys
sys.path.insert(0, "/content/cv-analyzer/backend")

from ocr_module import extrair_texto
from nlp_module import analisar_curriculo
from ranker import calcular_score
import gradio as gr

def processar(arquivo, descricao_vaga):
    if arquivo is None:            return "❌ Envie um currículo.", "", "", ""
    if not descricao_vaga.strip(): return "❌ Descreva a vaga.",    "", "", ""
    try:
        texto_bruto = extrair_texto(arquivo.name)
        if not texto_bruto:        return "❌ Não consegui extrair texto.", "", "", ""

        analise = analisar_curriculo(texto_bruto)
        r = calcular_score(analise["texto_limpo"], descricao_vaga, analise["habilidades"])

        principal = (
            f"## {r['classificacao']}\n"
            f"**Score:** {r['score']} / 100 — {r['recomendacao']}\n\n---\n"
            f"📊 Similaridade textual: **{r['sim_textual']}%** · "
            f"🛠️ Match de habilidades: **{r['match_hab']}%**"
        )
        habilidades = (
            f"**✅ Em comum com a vaga:** {', '.join(r['em_comum']) or 'Nenhuma'}\n\n"
            f"**➕ Extras do candidato:** {', '.join(r['extras']) or 'Nenhuma'}"
        )
        ent  = analise["entidades"]
        info = (
            f"**📧 E-mail:** {analise['email']}\n\n"
            f"**📞 Telefone:** {analise['telefone']}\n\n"
            f"**🏢 Organizações:** {', '.join(ent['organizacoes'][:5]) or 'Não identificadas'}\n\n"
            f"**📅 Datas:** {', '.join(ent['datas'][:5]) or 'Não identificadas'}"
        )
        ocr_preview = texto_bruto[:1500] + ("..." if len(texto_bruto) > 1500 else "")
        return principal, habilidades, info, ocr_preview

    except Exception as e:
        return f"❌ Erro: {e}", "", "", ""


with gr.Blocks(title="Analisador de Currículos", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🤖 Analisador de Currículos com OCR + IA")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📤 Entrada")
            arquivo_in = gr.File(label="Currículo (PDF, JPG ou PNG)",
                                 file_types=[".pdf", ".jpg", ".jpeg", ".png"])
            vaga_in    = gr.Textbox(label="Descrição da vaga",
                                    placeholder="Cole aqui os requisitos da vaga...",
                                    lines=8)
            btn        = gr.Button("🔍 Analisar", variant="primary", size="lg")
        with gr.Column():
            gr.Markdown("### 📊 Resultado")
            out_score = gr.Markdown()
            out_hab   = gr.Markdown()
            out_info  = gr.Markdown()
    with gr.Accordion("📄 Texto extraído pelo OCR", open=False):
        out_ocr = gr.Textbox(lines=10)
    btn.click(fn=processar,
              inputs=[arquivo_in, vaga_in],
              outputs=[out_score, out_hab, out_info, out_ocr])

app.launch(share=True)

/tmp/ipykernel_788/3463356740.py:44: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Analisador de Currículos", theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://464ca5d4f3b264dbca.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Célula 10 — Download do projeto completo

In [17]:
import shutil
from google.colab import files

os.chdir("/content")
shutil.make_archive("cv-analyzer", "zip", ".", "cv-analyzer")
files.download("cv-analyzer.zip")
print("✅ Download iniciado!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download iniciado!
